# Profiling y depuración

## 1.- Imports y configuración

In [1]:
import os
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# ============================================================
# RUTAS
# ============================================================

# Feather generado por el notebook 2024
RUTA_FEATHER = "licitaciones_anual_depurado.feather"

# Archivos auxiliares
RUTA_ORGANISMOS = "organismos_comuna.csv"
RUTA_COMUNAS_REGIONES = "comunas_regiones.csv"

# Salida final
RUTA_SALIDA = "LICIT_depurando_sinNulos.feather"

## 2.- Cargar Feather

In [2]:
df = pd.read_feather(RUTA_FEATHER)

print("=" * 80)
print("DATASET RECIBIDO")
print("=" * 80)

print(f"Filas: {len(df):,}")
print(f"Columnas: {len(df.columns):,}")
print(f"Dimensiones: {df.shape}")

print(
    f"Memoria: "
    f"{df.memory_usage(deep=True).sum() / 1024**2:,.2f} MB"
)

print("Dimensiones actuales:", df.shape)

print("\nNulos actuales:")
print(
    df[["MontoEstimado", "sector"]]
    .isna()
    .sum()
)

print("\nColumnas:")
print(df.columns.tolist())

display(df.head())

DATASET RECIBIDO
Filas: 2,260,107
Columnas: 16
Dimensiones: (2260107, 16)
Memoria: 176.75 MB
Dimensiones actuales: (2260107, 16)

Nulos actuales:
MontoEstimado    3302
sector           7817
dtype: int64

Columnas:
['CodigoOrganismo', 'sector', 'CodigoUnidad', 'ComunaUnidad', 'Tipo', 'Moneda Adquisicion', 'CantidadReclamos', 'FechaCierre', 'FechaPublicacion', 'MontoEstimado', 'NumeroOferentes', 'CodigoProductoONU', 'Cantidad', 'Cantidad Ofertada', 'Valor Total Ofertado', 'Oferta seleccionada']


,CodigoOrganismo,sector,CodigoUnidad,ComunaUnidad,Tipo,Moneda Adquisicion,CantidadReclamos,FechaCierre,FechaPublicacion,MontoEstimado,NumeroOferentes,CodigoProductoONU,Cantidad,Cantidad Ofertada,Valor Total Ofertado,Oferta seleccionada
0,6936,Salud,1604,Viña del Mar,LR,Peso Chileno,3521.0,2024-02-12,2024-01-11,645753500.0,10.0,42221504,60.0,60.0,486000.0,Seleccionada
1,6936,Salud,1604,Viña del Mar,LR,Peso Chileno,3521.0,2024-02-12,2024-01-11,645753500.0,10.0,42221504,60.0,60.0,486000.0,Seleccionada
2,6936,Salud,1604,Viña del Mar,LR,Peso Chileno,3521.0,2024-02-12,2024-01-11,645753500.0,10.0,42221504,60.0,60.0,599400.0,No Seleccionada
3,6936,Salud,1604,Viña del Mar,LR,Peso Chileno,3521.0,2024-02-12,2024-01-11,645753500.0,10.0,42221504,60.0,60.0,599400.0,No Seleccionada
4,6936,Salud,1604,Viña del Mar,LR,Peso Chileno,3521.0,2024-02-12,2024-01-11,645753500.0,10.0,42221504,60.0,60.0,486000.0,Seleccionada


## 3.- Variable objetivo

In [3]:
# ============================================================
# 2. VARIABLE OBJETIVO
# ============================================================

if "Seleccionada" not in df.columns:

    if "Oferta seleccionada" not in df.columns:
        raise KeyError(
            "No existe 'Oferta seleccionada' ni 'Seleccionada'."
        )

    valores = set(
        df["Oferta seleccionada"]
        .dropna()
        .unique()
    )

    esperados = {
        "Seleccionada",
        "No Seleccionada"
    }

    inesperados = valores - esperados

    if inesperados:
        raise ValueError(
            f"Valores inesperados en Oferta seleccionada: "
            f"{inesperados}"
        )

    # Eliminar registros sin target
    df = df.dropna(
        subset=["Oferta seleccionada"]
    ).copy()

    # Crear target binario
    df["Seleccionada"] = (
        df["Oferta seleccionada"]
        .map({
            "Seleccionada": 1,
            "No Seleccionada": 0
        })
    )

    df.drop(
        columns=["Oferta seleccionada"],
        inplace=True
    )


print("Distribución del target:\n")
print(df["Seleccionada"].value_counts())

print("\nPorcentaje:\n")

print(
    (
        df["Seleccionada"]
        .value_counts(normalize=True)
        * 100
    ).round(4)
)

Distribución del target:

Seleccionada
0    1901383
1     358724
Name: count, dtype: int64

Porcentaje:

Seleccionada
0    84.128
1    15.872
Name: proportion, dtype: float64


## 4.- Cargar relación organismo/comuna

In [4]:
# ============================================================
# 3. ORGANISMO -> COMUNA
# ============================================================

organismos_comuna = pd.read_csv(
    RUTA_ORGANISMOS
)

columnas_requeridas = {
    "codigo_organismo",
    "nombre_comuna"
}

faltantes = (
    columnas_requeridas
    - set(organismos_comuna.columns)
)

if faltantes:

    raise KeyError(
        f"Faltan columnas en organismos_comuna.csv: "
        f"{sorted(faltantes)}"
    )


# Verificar que un organismo tenga una sola comuna
if organismos_comuna[
    "codigo_organismo"
].duplicated().any():

    raise ValueError(
        "codigo_organismo contiene duplicados "
        "en organismos_comuna.csv"
    )


print(
    f"Organismos disponibles: "
    f"{len(organismos_comuna):,}"
)

Organismos disponibles: 1,168


## 5.- Completar comunas faltantes

In [5]:
# ============================================================
# COMPLETAR COMUNAS
# ============================================================

if "CodigoOrganismo" not in df.columns:
    raise KeyError(
        "El dataset no contiene CodigoOrganismo."
    )

if "ComunaUnidad" not in df.columns:
    raise KeyError(
        "El dataset no contiene ComunaUnidad."
    )


# ============================================================
# INFORMACIÓN ANTES DEL MERGE
# ============================================================

filas_antes = len(df)

comunas_antes = (
    df["ComunaUnidad"]
    .isna()
    .sum()
)

print(
    f"Tipo original de ComunaUnidad: "
    f"{df['ComunaUnidad'].dtype}"
)

print(
    f"Comunas faltantes antes: "
    f"{comunas_antes:,}"
)


# ============================================================
# MERGE ORGANISMO -> COMUNA
# ============================================================

df = df.merge(
    organismos_comuna[
        [
            "codigo_organismo",
            "nombre_comuna"
        ]
    ],
    how="left",
    left_on="CodigoOrganismo",
    right_on="codigo_organismo",
    validate="many_to_one"
)


# ============================================================
# VERIFICAR QUE EL MERGE NO DUPLICÓ FILAS
# ============================================================

if len(df) != filas_antes:
    raise RuntimeError(
        "El merge organismo-comuna "
        "modificó la cantidad de filas."
    )


# ============================================================
# COMPLETAR COMUNA
#
# ComunaUnidad viene como category desde el Feather.
# Se convierte temporalmente a string para permitir incorporar
# comunas que no estaban dentro de las categorías originales.
# ============================================================


df["ComunaUnidad"] = (
    df["ComunaUnidad"]
    .astype("string")
)

df["nombre_comuna"] = (
    df["nombre_comuna"]
    .astype("string")
)


df["ComunaUnidad"] = (
    df["ComunaUnidad"]
    .fillna(
        df["nombre_comuna"]
    )
)


# ============================================================
# RESULTADO
# ============================================================

comunas_despues = (
    df["ComunaUnidad"]
    .isna()
    .sum()
)


print(
    f"Comunas faltantes después: "
    f"{comunas_despues:,}"
)

print(
    f"Comunas recuperadas: "
    f"{comunas_antes - comunas_despues:,}"
)

print(
    f"Tipo actual de ComunaUnidad: "
    f"{df['ComunaUnidad'].dtype}"
)


# ============================================================
# ELIMINAR COLUMNAS AUXILIARES DEL MERGE
# ============================================================

df.drop(
    columns=[
        "codigo_organismo",
        "nombre_comuna"
    ],
    inplace=True
)

Tipo original de ComunaUnidad: category
Comunas faltantes antes: 152,280
Comunas faltantes después: 7,927
Comunas recuperadas: 144,353
Tipo actual de ComunaUnidad: string


## 6.- Normalizar comunas

In [6]:
# ============================================================
# 4. COMUNA -> REGIÓN
# ============================================================

def normalizar_comuna(valor):

    if pd.isna(valor):
        return pd.NA

    valor = str(valor)

    valor = (
        valor
        .strip()
        .lower()
    )

    # Quitar tildes
    valor = unicodedata.normalize(
        "NFD",
        valor
    )

    valor = "".join(
        c
        for c in valor
        if unicodedata.category(c) != "Mn"
    )

    # Limpiar caracteres
    valor = re.sub(
        r"[^a-z0-9]+",
        " ",
        valor
    )

    valor = re.sub(
        r"\s+",
        " ",
        valor
    ).strip()

    return valor

## 7.- Cargar catálogo comuna/región

In [7]:
comunas_regiones = pd.read_csv(
    RUTA_COMUNAS_REGIONES
)


columnas_requeridas = {
    "nombre_comuna",
    "nombre_region"
}


faltantes = (

    columnas_requeridas

    - set(
        comunas_regiones.columns
    )
)


if faltantes:

    raise KeyError(
        f"Faltan columnas en comunas_regiones.csv: "
        f"{sorted(faltantes)}"
    )


# Normalizar comuna dataset

df["_comuna_normalizada"] = (

    df["ComunaUnidad"]

    .apply(
        normalizar_comuna
    )
)


# Normalizar catálogo

comunas_regiones[
    "_comuna_normalizada"
] = (

    comunas_regiones[
        "nombre_comuna"
    ]

    .apply(
        normalizar_comuna
    )
)

## 8.- Alias de comunas

In [8]:
# ============================================================
# ALIAS DE COMUNAS
# ============================================================

alias_comunas = {

    "santiago centro":
        "santiago",

    "la calera":
        "calera",

    "puerto saavedra":
        "saavedra",

    "puerto natales":
        "natales",

    "san vicente de tagua tagua":
        "san vicente",

    "til til":
        "tiltil",

    "alto bio bio":
        "alto biobio",

    "chol chol":
        "cholchol",

    "torres del payne":
        "torres del paine",

    "las guaitecas":
        "guaitecas"
}


df["_comuna_normalizada"] = (

    df["_comuna_normalizada"]

    .replace(
        alias_comunas
    )
)


# Verificar que el catálogo siga siendo único

if comunas_regiones[
    "_comuna_normalizada"
].duplicated().any():

    raise ValueError(
        "El catálogo comuna-región contiene "
        "comunas duplicadas después de normalizar."
    )

## 9.- Asignar región

In [9]:
# ============================================================
# MERGE COMUNA -> REGIÓN
# ============================================================

filas_antes = len(df)


df = df.merge(

    comunas_regiones[
        [
            "_comuna_normalizada",
            "nombre_region"
        ]
    ],

    how="left",

    on="_comuna_normalizada",

    validate="many_to_one"
)


if len(df) != filas_antes:

    raise RuntimeError(
        "El merge comuna-región "
        "modificó la cantidad de filas."
    )


df.rename(
    columns={
        "nombre_region":
        "Region"
    },
    inplace=True
)


# Corrección explícita del notebook original

df.loc[
    df["ComunaUnidad"].eq("Trehuaco"),
    "Region"
] = "Región de Ñuble"


print(
    "Registros sin región:",
    df["Region"].isna().sum()
)

print(f"FILAS ACTUALES: {len(df):,}")

Registros sin región: 7927
FILAS ACTUALES: 2,260,107


## 10.- Eliminar registros sin región

In [10]:
# ============================================================
# FILTRAR REGIÓN
# ============================================================

filas_antes = len(df)

df = df[
    df["Region"].notna()
    &
    df["Region"].ne(
        "Desconocida"
    )
].copy()


print(
    "Filas eliminadas:",
    filas_antes - len(df)
)


print(f"FILAS ACTUALES: {len(df):,}")

Filas eliminadas: 7927
FILAS ACTUALES: 2,252,180


## 11.- Diagnóstico a MontoEstimado

In [11]:
# ============================================================
# DIAGNÓSTICO ANTES DE CONVERSIÓN NUMÉRICA
# ============================================================

print("Dimensiones actuales:", df.shape)

print("\nNulos actuales:")
print(
    df[["MontoEstimado", "sector"]]
    .isna()
    .sum()
)

# ------------------------------------------------------------
# Diagnosticar valores de MontoEstimado que no son convertibles
# ------------------------------------------------------------

original = df["MontoEstimado"].astype("string")

convertido_directo = pd.to_numeric(
    original,
    errors="coerce"
)

problemas = (
    original.notna()
    & convertido_directo.isna()
)

print(
    "\nValores NO nulos que pd.to_numeric no entiende:",
    problemas.sum()
)

print("\nEjemplos de valores problemáticos:")
print(
    original[problemas]
    .value_counts()
    .head(30)
)

Dimensiones actuales: (2252180, 18)

Nulos actuales:
MontoEstimado    3301
sector           7817
dtype: int64

Valores NO nulos que pd.to_numeric no entiende: 0

Ejemplos de valores problemáticos:
Series([], Name: count, dtype: int64[pyarrow])


## 11.a Eliminar Valores NaN de MontoEstimado

In [12]:
# ============================================================
# 11.- LIMPIEZA BÁSICA
# Eliminar registros sin MontoEstimado
# ============================================================

filas_antes = len(df)

print("Filas antes:", filas_antes)
print(
    "MontoEstimado NaN:",
    df["MontoEstimado"].isna().sum()
)

df = df.dropna(
    subset=["MontoEstimado"]
).copy()

filas_despues = len(df)

print("\nFilas eliminadas:", filas_antes - filas_despues)
print("Filas después:", filas_despues)
print("Dimensiones:", df.shape)

# Verificación
print(
    "MontoEstimado NaN restantes:",
    df["MontoEstimado"].isna().sum()
)

Filas antes: 2252180
MontoEstimado NaN: 3301

Filas eliminadas: 3301
Filas después: 2248879
Dimensiones: (2248879, 18)
MontoEstimado NaN restantes: 0


## 11.b- ELIMINAR COLUMNAS AUXILIARES DE GEOGRAFÍA

In [13]:
# ============================================================
# ELIMINAR COLUMNAS AUXILIARES DE GEOGRAFÍA
# ============================================================

columnas_eliminar = [
    "CodigoOrganismo",
    "ComunaUnidad",
    "_comuna_normalizada"
]

df.drop(
    columns=[
        col for col in columnas_eliminar
        if col in df.columns
    ],
    inplace=True
)

print(
    "Columnas auxiliares eliminadas:",
    columnas_eliminar
)

print(
    "Dimensiones actuales:",
    df.shape
)

Columnas auxiliares eliminadas: ['CodigoOrganismo', 'ComunaUnidad', '_comuna_normalizada']
Dimensiones actuales: (2248879, 15)


## 12.- Convertir columnas numéricas

In [14]:
# ============================================================
# CONVERSIÓN NUMÉRICA
# ============================================================

columnas_numericas = [
    "MontoEstimado",
    "Cantidad",
    "Cantidad Ofertada",
    "Valor Total Ofertado"
]


for col in columnas_numericas:
    if col not in df.columns:
        raise KeyError(
            f"Falta la columna: {col}"
        )

    # Solo hacer reemplazo decimal
    # cuando no sea ya numérica

    if not pd.api.types.is_numeric_dtype(df[col]):
        df[col] = (
            df[col]
            .astype("string")

            .str.replace(
                ",",
                ".",
                regex=False
            )
        )

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

resumen_numericas = pd.DataFrame({

    "Nulos":
        df[columnas_numericas]
        .isna()
        .sum(),

    "Tipo":
        df[columnas_numericas]
        .dtypes
        .astype(str)
})

display(
    resumen_numericas
)

,Nulos,Tipo
MontoEstimado,0,float64
Cantidad,0,float64
Cantidad Ofertada,0,float64
Valor Total Ofertado,0,float64


### 12.1.- Diagnóstico de Nulos

In [15]:
# ============================================================
# DIAGNÓSTICO DE NULOS EN VARIABLES NUMÉRICAS
# ============================================================

columnas_revisar = [
    "Cantidad",
    "Cantidad Ofertada",
    "Valor Total Ofertado"
]

# Verificar existencia
faltantes = [
    col
    for col in columnas_revisar
    if col not in df.columns
]

if faltantes:
    raise KeyError(
        f"No existen las siguientes columnas: {faltantes}"
    )


# ------------------------------------------------------------
# Cantidad y porcentaje de nulos
# ------------------------------------------------------------

resumen_nulos = pd.DataFrame({
    "Nulos": (
        df[columnas_revisar]
        .isna()
        .sum()
    ),
    "Porcentaje": (
        df[columnas_revisar]
        .isna()
        .mean()
        .mul(100)
        .round(4)
    )
})

print("NULOS POR COLUMNA")
print("=" * 60)

display(
    resumen_nulos
)


# ------------------------------------------------------------
# Patrones de nulos
# ------------------------------------------------------------

print("\nPATRONES DE NULOS")
print("=" * 60)

patrones_nulos = (
    df[columnas_revisar]
    .isna()
    .value_counts()
    .rename("Filas")
    .reset_index()
)

display(
    patrones_nulos
)


# ------------------------------------------------------------
# Cantidad de filas afectadas por al menos un NaN
# ------------------------------------------------------------

filas_con_nulos = (
    df[columnas_revisar]
    .isna()
    .any(axis=1)
    .sum()
)

print(
    f"\nFilas con al menos un NaN "
    f"en estas variables: {filas_con_nulos:,}"
)

print(
    f"Porcentaje del dataset: "
    f"{filas_con_nulos / len(df) * 100:.4f}%"
)

NULOS POR COLUMNA


,Nulos,Porcentaje
Cantidad,0,0.0
Cantidad Ofertada,0,0.0
Valor Total Ofertado,0,0.0



PATRONES DE NULOS


,Cantidad,Cantidad Ofertada,Valor Total Ofertado,Filas
0,False,False,False,2248879



Filas con al menos un NaN en estas variables: 0
Porcentaje del dataset: 0.0000%


## 13.- Revisar monedas

In [16]:
# ============================================================
# 6. MONEDAS
# ============================================================

print(
    df[
        "Moneda Adquisicion"
    ]
    .value_counts(
        dropna=False
    )
)

Moneda Adquisicion
Peso Chileno         2235698
Dolar                   7214
Unidad de Fomento       5582
Moneda revisar           355
Euro                      30
Name: count, dtype: int64


## 14.- Filtrar monedas soportadas

In [17]:
monedas_soportadas = {
    "Peso Chileno",
    "Dolar",
    "Unidad de Fomento"
}


mask_no_soportada = (

    ~df[
        "Moneda Adquisicion"
    ]

    .isin(
        monedas_soportadas
    )
)


print(
    "Filas con moneda no soportada:",
    mask_no_soportada.sum()
)

if mask_no_soportada.any():

    print(

        df.loc[
            mask_no_soportada,
            "Moneda Adquisicion"
        ]

        .value_counts(
            dropna=False
        )
    )

df = (

    df.loc[
        ~mask_no_soportada
    ]
    .copy()
)

Filas con moneda no soportada: 385
Moneda Adquisicion
Moneda revisar       355
Euro                  30
Dolar                  0
Peso Chileno           0
Unidad de Fomento      0
Name: count, dtype: int64


## 15.- Banco Central

In [18]:
# ============================================================
# BANCO CENTRAL
# ============================================================

import bcchapi

TOKEN = "$2a$10$/qrcAjqKmwOEw8YTjGnAmuaQfbSkZNw6DjnY8/lVD82z01MQJyXUi"

siete = bcchapi.Siete(
    token=TOKEN
)

print("Cliente del Banco Central configurado correctamente.")

Cliente del Banco Central configurado correctamente.


## 16.- Rango de fechas

In [19]:
df["FechaPublicacion"] = pd.to_datetime(

    df["FechaPublicacion"],

    errors="coerce"
)


if df[
    "FechaPublicacion"
].isna().any():

    raise ValueError(
        "Existen FechaPublicacion inválidas."
    )


fecha_inicio = (

    df["FechaPublicacion"]

    .min()

    .strftime(
        "%Y-%m-%d"
    )
)


fecha_fin = (

    df["FechaPublicacion"]

    .max()

    .strftime(
        "%Y-%m-%d"
    )
)


print(
    "Fecha inicial:",
    fecha_inicio
)

print(
    "Fecha final:",
    fecha_fin
)

Fecha inicial: 2024-01-02
Fecha final: 2024-12-31


## 17.- Descargar tasas

In [20]:
tasas = siete.cuadro(

    series=[

        # Dólar observado
        "F073.TCO.PRE.Z.D",

        # UF
        "F073.UFF.PRE.Z.D"
    ],

    desde=fecha_inicio,

    hasta=fecha_fin,

    nombres=[
        "USD_CLP",
        "UF_CLP"
    ]
)


tasas = (
    tasas
    .reset_index()
)


# El nombre del índice puede variar.
# Tomamos explícitamente la primera columna.

tasas.rename(

    columns={
        tasas.columns[0]:
        "FechaPublicacion"
    },

    inplace=True
)


tasas[
    "FechaPublicacion"
] = pd.to_datetime(

    tasas[
        "FechaPublicacion"
    ]
)


tasas = tasas.sort_values(
    "FechaPublicacion"
)


display(
    tasas.head()
)

,FechaPublicacion,USD_CLP,UF_CLP
0,2024-01-02,877.12,36805.92
1,2024-01-03,880.92,36814.21
2,2024-01-04,884.39,36822.49
3,2024-01-05,884.45,36830.78
4,2024-01-06,NaN,36839.07


## 18.- Completar tasas faltantes

In [21]:
# ============================================================
# COMPLETAR FINES DE SEMANA / FERIADOS
# ============================================================

tasas[
    [
        "USD_CLP",
        "UF_CLP"
    ]
] = (

    tasas[
        [
            "USD_CLP",
            "UF_CLP"
        ]
    ]
    .ffill()
)

print(
    tasas[
        [
            "USD_CLP",
            "UF_CLP"
        ]
    ]
    .isna()
    .sum()
)

USD_CLP    0
UF_CLP     0
dtype: int64


## 19.- Merge de tasas

In [22]:
filas_antes = len(df)

df = df.merge(

    tasas[
        [
            "FechaPublicacion",
            "USD_CLP",
            "UF_CLP"
        ]
    ],

    on="FechaPublicacion",

    how="left",

    validate="many_to_one"
)


if len(df) != filas_antes:

    raise RuntimeError(
        "El merge de tasas modificó "
        "la cantidad de filas."
    )

print(
    df[
        [
            "USD_CLP",
            "UF_CLP"
        ]
    ]
    .isna()
    .sum()
)

USD_CLP    0
UF_CLP     0
dtype: int64


## 20.- Validar tasas

In [23]:
if df[
    [
        "USD_CLP",
        "UF_CLP"
    ]
].isna().any().any():

    raise ValueError(
        "Existen registros sin tasa "
        "después del merge."
    )

## 21.- Conversión a CLP

In [24]:
# ============================================================
# CONVERTIR MONTOS A CLP
# ============================================================

factor_clp = np.select(

    [

        df[
            "Moneda Adquisicion"
        ].eq(
            "Peso Chileno"
        ),

        df[
            "Moneda Adquisicion"
        ].eq(
            "Dolar"
        ),

        df[
            "Moneda Adquisicion"
        ].eq(
            "Unidad de Fomento"
        )
    ],

    [

        1.0,

        df["USD_CLP"],

        df["UF_CLP"]
    ],

    default=np.nan
)


df[
    "MontoEstimado"
] = (

    df[
        "MontoEstimado"
    ]

    .astype(
        "float64"
    )

    * factor_clp
)


df[
    "Valor Total Ofertado"
] = (

    df[
        "Valor Total Ofertado"
    ]

    .astype(
        "float64"
    )

    * factor_clp
)

## 22.- Eliminar columnas auxiliares de moneda

In [25]:
df.drop(

    columns=[

        "USD_CLP",

        "UF_CLP",

        "Moneda Adquisicion"
    ],

    inplace=True
)


print(
    "Nulos MontoEstimado:",
    df["MontoEstimado"].isna().sum()
)

print(
    "Nulos Valor Total Ofertado:",
    df["Valor Total Ofertado"].isna().sum()
)

Nulos MontoEstimado: 0
Nulos Valor Total Ofertado: 0


### 22.1.- diagnóstico POST conversión monetaria

In [26]:
# ============================================================
# REVISAR NULOS DESPUÉS DE CONVERSIÓN MONETARIA
# ============================================================

columnas_obligatorias = [
    "Cantidad",
    "Cantidad Ofertada",
    "Valor Total Ofertado"
]

resumen_post_moneda = pd.DataFrame({
    "Nulos": (
        df[columnas_obligatorias]
        .isna()
        .sum()
    ),
    "Porcentaje": (
        df[columnas_obligatorias]
        .isna()
        .mean()
        .mul(100)
        .round(4)
    )
})

print("NULOS DESPUÉS DE CONVERSIÓN MONETARIA")
print("=" * 60)

display(
    resumen_post_moneda
)


patrones_post_moneda = (
    df[columnas_obligatorias]
    .isna()
    .value_counts()
    .rename("Filas")
    .reset_index()
)

print("\nPATRONES:")
display(
    patrones_post_moneda
)

NULOS DESPUÉS DE CONVERSIÓN MONETARIA


,Nulos,Porcentaje
Cantidad,0,0.0
Cantidad Ofertada,0,0.0
Valor Total Ofertado,0,0.0



PATRONES:


,Cantidad,Cantidad Ofertada,Valor Total Ofertado,Filas
0,False,False,False,2248494


### 22.2.- Eliminar registros incompletos

In [27]:
# ============================================================
# ELIMINAR REGISTROS INCOMPLETOS
# ============================================================

columnas_obligatorias = [
    "Cantidad",
    "Cantidad Ofertada",
    "Valor Total Ofertado"
]


# ------------------------------------------------------------
# Estado antes
# ------------------------------------------------------------

filas_antes = len(df)

filas_problematicas = (
    df[columnas_obligatorias]
    .isna()
    .any(axis=1)
    .sum()
)


print("=" * 60)
print("ELIMINACIÓN DE REGISTROS INCOMPLETOS")
print("=" * 60)

print(
    f"Filas antes:       {filas_antes:,}"
)

print(
    f"Filas incompletas: {filas_problematicas:,}"
)


# ------------------------------------------------------------
# Eliminar
# ------------------------------------------------------------

df = (
    df
    .dropna(
        subset=columnas_obligatorias
    )
    .copy()
)


# ------------------------------------------------------------
# Estado después
# ------------------------------------------------------------

filas_despues = len(df)

print(
    f"Filas después:     {filas_despues:,}"
)

print(
    f"Filas eliminadas:  "
    f"{filas_antes - filas_despues:,}"
)

print(
    f"Porcentaje eliminado: "
    f"{(filas_antes - filas_despues) / filas_antes * 100:.4f}%"
)


# ------------------------------------------------------------
# Verificación
# ------------------------------------------------------------

nulos_restantes = (
    df[columnas_obligatorias]
    .isna()
    .sum()
)

print("\nNulos restantes:")

print(
    nulos_restantes
)

ELIMINACIÓN DE REGISTROS INCOMPLETOS
Filas antes:       2,248,494
Filas incompletas: 0
Filas después:     2,248,494
Filas eliminadas:  0
Porcentaje eliminado: 0.0000%

Nulos restantes:
Cantidad                0
Cantidad Ofertada       0
Valor Total Ofertado    0
dtype: int64


## 23.- Fechas

In [28]:
# ============================================================
# 7. FECHAS
# ============================================================

df[
    "FechaCierre"
] = pd.to_datetime(

    df[
        "FechaCierre"
    ],

    errors="coerce"
)


df[
    "FechaPublicacion"
] = pd.to_datetime(

    df[
        "FechaPublicacion"
    ],

    errors="coerce"
)

## 24.- Duración del proceso

In [29]:
df[
    "DuracionProceso"
] = (

    df[
        "FechaCierre"
    ]

    - df[
        "FechaPublicacion"
    ]
).dt.days


print(
    df["DuracionProceso"]
    .describe()
    .round(2)
    .apply(lambda x: f"{x:,.2f}")
)

# Verificar si hay duraciones negativas.
print(
    "\nDuraciones negativas:",
    (
        df[
            "DuracionProceso"
        ]
        < 0
    ).sum()
)

count    2,248,494.00
mean            11.68
std              7.62
min              3.00
25%              7.00
50%             10.00
75%             13.00
max            215.00
Name: DuracionProceso, dtype: str

Duraciones negativas: 0


## 25.- Eliminar duraciones inválidas

In [30]:
filas_antes = len(df)

df = df[

    df[
        "DuracionProceso"
    ]
    .notna()

    &

    df[
        "DuracionProceso"
    ]
    .ge(0)

].copy()


print(
    "Filas eliminadas por duración inválida:",
    filas_antes - len(df)
)

Filas eliminadas por duración inválida: 0


## 26.- Variables temporales

In [31]:
df[
    "AnioPublicacion"
] = (

    df[
        "FechaPublicacion"
    ]

    .dt.year
)


df[
    "MesPublicacion"
] = (

    df[
        "FechaPublicacion"
    ]

    .dt.month
)


df[
    "DiaSemanaPublicacion"
] = (

    df[
        "FechaPublicacion"
    ]

    .dt.dayofweek
)


df.drop(

    columns=[

        "FechaCierre",

        "FechaPublicacion"
    ],

    inplace=True
)

## 27.- Revisar categóricas

In [32]:
# ============================================================
# 8. VARIABLES CATEGÓRICAS
# ============================================================

columnas_categoricas = [

    "sector",

    "Tipo",

    "Region"
]


for col in columnas_categoricas:

    print(
        f"\n--- {col} ---"
    )

    print(
        df[col]
        .value_counts(
            dropna=False
        )
    )


--- sector ---
sector
Municipalidades                1135218
Salud                           555102
Gob. Central, Universidades     230890
FFAA                            167093
Otros                            98987
Obras Públicas                   46545
NaN                               7817
Legislativo y judicial            6842
Name: count, dtype: int64

--- Tipo ---
Tipo
LE    1018403
L1     775149
LQ     189694
LP     137425
LR     119677
O1       5696
CO       1292
E2        563
B2        173
H2        173
LS        140
I2         73
O2         36
Name: count, dtype: int64

--- Region ---
Region
Región Metropolitana de Santiago                       684846
Región del Biobío                                      244523
Región de Valparaíso                                   205171
Región del Maule                                       193741
Región de La Araucanía                                 160280
Región de Los Lagos                                    146922
Región del Libert

## 28.- One-Hot Encoding

In [33]:
# ============================================================
# ONE-HOT ENCODING
# ============================================================

df = pd.get_dummies(

    df,

    columns=columnas_categoricas,

    drop_first=False,

    dtype=np.int8
)


print(
    "Dimensiones después del encoding:",
    df.shape
)

Dimensiones después del encoding: (2248494, 49)


## 29.- Función de conversión segura

In [34]:
# ============================================================
# FUNCIÓN DE CONVERSIÓN SEGURA A ENTEROS
# ============================================================

def convertir_entero_seguro(
    df,
    columna,
    tipo
):

    # --------------------------------------------------------
    # 1. Verificar existencia
    # --------------------------------------------------------

    if columna not in df.columns:

        raise KeyError(
            f"No existe la columna: {columna}"
        )


    # --------------------------------------------------------
    # 2. Guardar serie original
    # --------------------------------------------------------

    serie_original = df[columna]


    # --------------------------------------------------------
    # 3. Convertir temporalmente a numérico
    #
    # Permite trabajar también con columnas que vienen
    # como category desde el Feather anterior.
    # --------------------------------------------------------

    serie_numerica = pd.to_numeric(
        serie_original.astype("string"),
        errors="coerce"
    )


    # --------------------------------------------------------
    # 4. Detectar valores que no pudieron convertirse
    # --------------------------------------------------------

    invalidos = (
        serie_original.notna()
        & serie_numerica.isna()
    )


    if invalidos.any():

        ejemplos = (
            serie_original[
                invalidos
            ]
            .astype("string")
            .unique()[:10]
        )

        raise ValueError(
            f"{columna} contiene valores no numéricos. "
            f"Ejemplos: {list(ejemplos)}"
        )


    # --------------------------------------------------------
    # 5. No permitir NaN
    # --------------------------------------------------------

    if serie_numerica.isna().any():

        cantidad_nulos = (
            serie_numerica
            .isna()
            .sum()
        )

        raise ValueError(
            f"{columna} contiene "
            f"{cantidad_nulos:,} valores NaN."
        )


    # --------------------------------------------------------
    # 6. Verificar que todos sean enteros
    # --------------------------------------------------------

    no_enteros = (
        serie_numerica
        != np.floor(
            serie_numerica
        )
    )


    if no_enteros.any():

        ejemplos = (
            serie_numerica[
                no_enteros
            ]
            .unique()[:10]
        )

        raise ValueError(
            f"{columna} contiene valores decimales "
            f"que no pueden convertirse a entero "
            f"sin perder información. "
            f"Ejemplos: {list(ejemplos)}"
        )


    # --------------------------------------------------------
    # 7. Rango observado
    # --------------------------------------------------------

    minimo = (
        serie_numerica
        .min()
    )

    maximo = (
        serie_numerica
        .max()
    )


    # --------------------------------------------------------
    # 8. Rango permitido por dtype
    # --------------------------------------------------------

    limite = np.iinfo(
        np.dtype(tipo)
    )


    # --------------------------------------------------------
    # 9. Verificar overflow
    # --------------------------------------------------------

    if (
        minimo < limite.min
        or
        maximo > limite.max
    ):

        raise OverflowError(
            f"{columna} no cabe en {tipo}. "
            f"Rango observado: "
            f"[{minimo}, {maximo}]. "
            f"Rango permitido: "
            f"[{limite.min}, {limite.max}]"
        )


    # --------------------------------------------------------
    # 10. Conversión definitiva
    # --------------------------------------------------------

    df[columna] = (
        serie_numerica
        .astype(tipo)
    )


    # --------------------------------------------------------
    # 11. Información
    # --------------------------------------------------------

    print(
        f"{columna:<30} "
        f"{str(serie_original.dtype):<15} "
        f"-> {tipo:<8} "
        f"rango [{minimo}, {maximo}]"
    )

## 30.- Aplicar tipos

In [35]:
# ============================================================
# APLICAR TIPOS
# ============================================================


# ------------------------------------------------------------
# Memoria antes de la conversión
# ------------------------------------------------------------

memoria_antes = (
    df.memory_usage(
        deep=True
    )
    .sum()
    / 1024**2
)

print(
    f"Memoria antes de optimizar: "
    f"{memoria_antes:,.2f} MB"
)


# ============================================================
# INT8
# ============================================================

columnas_int8 = [
    "DiaSemanaPublicacion",
    "MesPublicacion",
    "Seleccionada"
]


for col in columnas_int8:

    convertir_entero_seguro(
        df,
        col,
        "int8"
    )


# ============================================================
# INT16
# ============================================================

columnas_int16 = [
    "AnioPublicacion",
    "CantidadReclamos",
    "DuracionProceso",
    "NumeroOferentes"
]


for col in columnas_int16:

    convertir_entero_seguro(
        df,
        col,
        "int16"
    )


# ============================================================
# INT32
# ============================================================

convertir_entero_seguro(
    df,
    "CodigoProductoONU",
    "int32"
)


print(
    "\nConversión de tipos completada."
)

Memoria antes de optimizar: 242.31 MB
DiaSemanaPublicacion           int32           -> int8     rango [0, 6]
MesPublicacion                 int32           -> int8     rango [1, 12]
Seleccionada                   category        -> int8     rango [0, 1]
AnioPublicacion                int32           -> int16    rango [2024, 2024]
CantidadReclamos               float64         -> int16    rango [0.0, 11352.0]
DuracionProceso                int64           -> int16    rango [3, 215]
NumeroOferentes                float64         -> int16    rango [0.0, 578.0]
CodigoProductoONU              uint32          -> int32    rango [10101501, 104102023]

Conversión de tipos completada.


## 31.- Resultado de memoria

In [36]:
# ============================================================
# RESULTADO DE OPTIMIZACIÓN DE MEMORIA
# ============================================================

memoria_despues = (
    df.memory_usage(
        deep=True
    )
    .sum()
    / 1024**2
)


reduccion_mb = (
    memoria_antes
    - memoria_despues
)


reduccion_porcentaje = (
    reduccion_mb
    / memoria_antes
    * 100
)


print("=" * 60)

print(
    f"Memoria antes:   "
    f"{memoria_antes:,.2f} MB"
)

print(
    f"Memoria después: "
    f"{memoria_despues:,.2f} MB"
)

print(
    f"Reducción:       "
    f"{reduccion_mb:,.2f} MB"
)

print(
    f"Reducción %:     "
    f"{reduccion_porcentaje:.2f}%"
)

print("=" * 60)

Memoria antes:   242.31 MB
Memoria después: 186.56 MB
Reducción:       55.75 MB
Reducción %:     23.01%


## 32.- Validación final

In [37]:
# ============================================================
# VALIDACIÓN FINAL DEL DATASET
# ============================================================

print("=" * 70)
print("VALIDACIÓN FINAL DEL DATASET")
print("=" * 70)

print(
    f"Filas:    {len(df):,}"
)

print(
    f"Columnas: {df.shape[1]:,}"
)

print(
    f"Dimensiones: {df.shape}"
)


# ============================================================
# 1. NULOS
# ============================================================

nulos = (
    df.isna()
    .sum()
)

nulos = (
    nulos[
        nulos > 0
    ]
    .sort_values(
        ascending=False
    )
)


if not nulos.empty:

    print(
        "\nERROR: existen valores nulos:"
    )

    print(
        nulos
    )

    raise ValueError(
        "El dataset todavía contiene "
        "valores nulos."
    )

else:

    print(
        "\n✓ No existen valores nulos."
    )


# ============================================================
# 2. COLUMNAS QUE NO DEBEN LLEGAR AL DATASET FINAL
# ============================================================

columnas_no_permitidas = [
    "ComunaUnidad",
    "_comuna_normalizada",
    "USD_CLP",
    "UF_CLP",
    "Moneda Adquisicion",
    "FechaPublicacion",
    "FechaCierre",
    "Oferta seleccionada"
]


columnas_presentes = [
    col
    for col in columnas_no_permitidas
    if col in df.columns
]


if columnas_presentes:

    raise ValueError(
        "Quedaron columnas auxiliares "
        "que deberían haber sido eliminadas: "
        f"{columnas_presentes}"
    )

else:

    print(
        "✓ No quedaron columnas auxiliares."
    )


# ============================================================
# 3. TARGET
# ============================================================

if "Seleccionada" not in df.columns:

    raise KeyError(
        "No existe la variable objetivo "
        "'Seleccionada'."
    )


valores_target = set(
    df[
        "Seleccionada"
    ]
    .unique()
    .tolist()
)


if not valores_target.issubset(
    {0, 1}
):

    raise ValueError(
        "Seleccionada contiene valores "
        f"inesperados: {valores_target}"
    )

else:

    print(
        "✓ Target Seleccionada contiene "
        "solamente 0 y 1."
    )


# ============================================================
# 4. DUPLICADOS COMPLETOS
# ============================================================

duplicados = (
    df.duplicated()
    .sum()
)

print(
    f"Duplicados completos: "
    f"{duplicados:,}"
)


# ============================================================
# 5. INFINITOS NUMÉRICOS
# ============================================================

columnas_numericas = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
)


infinitos = (
    np.isinf(
        df[
            columnas_numericas
        ]
    )
    .sum()
    .sum()
)


if infinitos > 0:

    raise ValueError(
        f"El dataset contiene "
        f"{infinitos:,} valores infinitos."
    )

else:

    print(
        "✓ No existen valores infinitos."
    )


print(
    "\n" + "=" * 70
)

print(
    "DATASET VALIDADO CORRECTAMENTE"
)

print(
    "=" * 70
)

VALIDACIÓN FINAL DEL DATASET
Filas:    2,248,494
Columnas: 49
Dimensiones: (2248494, 49)

✓ No existen valores nulos.
✓ No quedaron columnas auxiliares.
✓ Target Seleccionada contiene solamente 0 y 1.
Duplicados completos: 412,088
✓ No existen valores infinitos.

DATASET VALIDADO CORRECTAMENTE


## 33.- Revisar target

In [38]:
# ============================================================
# DISTRIBUCIÓN FINAL DEL TARGET
# ============================================================

distribucion_target = pd.DataFrame({
    "Cantidad": (
        df["Seleccionada"]
        .value_counts()
        .sort_index()
    ),
    "Porcentaje": (
        df["Seleccionada"]
        .value_counts(
            normalize=True
        )
        .sort_index()
        .mul(100)
        .round(4)
    )
})


print(
    "DISTRIBUCIÓN DEL TARGET"
)

print("=" * 60)

display(
    distribucion_target
)

DISTRIBUCIÓN DEL TARGET


,Cantidad,Porcentaje
Seleccionada,,
0,1891337,84.1157
1,357157,15.8843


## 35.- Tipos finales

In [39]:
# ============================================================
# TIPOS FINALES
# ============================================================

print("=" * 70)
print("TIPOS FINALES")
print("=" * 70)

print(
    df.dtypes
)


print(
    "\nResumen por dtype:"
)

print(
    df.dtypes
    .astype(str)
    .value_counts()
)


print(
    "\nDimensiones finales:"
)

print(
    df.shape
)

TIPOS FINALES
CodigoUnidad                                                   uint32
CantidadReclamos                                                int16
MontoEstimado                                                 float64
NumeroOferentes                                                 int16
CodigoProductoONU                                               int32
Cantidad                                                      float64
Cantidad Ofertada                                             float64
Valor Total Ofertado                                          float64
Seleccionada                                                     int8
DuracionProceso                                                 int16
AnioPublicacion                                                 int16
MesPublicacion                                                   int8
DiaSemanaPublicacion                                             int8
sector_FFAA                                                      int8
sector

## 36.- Guardar Feather final

In [40]:
# ============================================================
# GUARDAR DATASET FINAL
# ============================================================

RUTA_SALIDA = (
    "LICIT_depurando_sinNulos.feather"
)


# ------------------------------------------------------------
# Última protección antes de guardar
# ------------------------------------------------------------

nulos_totales = (
    df.isna()
    .sum()
    .sum()
)


if nulos_totales != 0:

    raise ValueError(
        f"No se guardará el Feather porque "
        f"todavía existen "
        f"{nulos_totales:,} valores nulos."
    )


# ------------------------------------------------------------
# Limpiar índice
# ------------------------------------------------------------

df.reset_index(
    drop=True,
    inplace=True
)


# ------------------------------------------------------------
# Guardar
# ------------------------------------------------------------

df.to_feather(
    RUTA_SALIDA
)


print("=" * 70)

print(
    "FEATHER GENERADO CORRECTAMENTE"
)

print("=" * 70)

print(
    f"Archivo: {RUTA_SALIDA}"
)

print(
    f"Filas: {len(df):,}"
)

print(
    f"Columnas: {df.shape[1]:,}"
)

print(
    f"Nulos: {df.isna().sum().sum():,}"
)

FEATHER GENERADO CORRECTAMENTE
Archivo: LICIT_depurando_sinNulos.feather
Filas: 2,248,494
Columnas: 49
Nulos: 0
